# 🧬 Notebook 01: Automated PubChem Curation & RDKit Database Builder (Database v1)
**Features Covered:** D1, D2, D3, E2

This notebook fetches Canonical SMILES and PubChem CIDs for our literature-curated *Cynodon dactylon* phytochemicals and Positive Controls, calculates Lipinski descriptors via RDKit, and exports `durva_compounds_v1.csv`.


In [ ]:
!pip install -q rdkit pubchempy pandas
import pandas as pd
import pubchempy as pcp
from rdkit import Chem
from rdkit.Chem import Descriptors


In [ ]:
compounds = [
    ('Luteolin', '10.30574/wjbphs.2025.24.2.0992', 'HPLC / HPTLC'),
    ('Apigenin', '10.1016/j.jep.2009.02.036', 'RP-HPLC'),
    ('Orientin', '10.47552/ijam.v16i1.5308', 'HPTLC'),
    ('Vitexin', '10.47552/ijam.v16i1.5308', 'HPTLC'),
    ('p-Coumaric acid', '10.1016/j.phyplu.2021.100020', 'RP-HPLC-PDA (0.48%)'),
    ('trans-Ferulic acid', '10.4314/bcse.v38i6.3', 'RP-HPLC-PDA (0.35%)'),
    ('Pinoresinol dimethyl ether', '10.70066/jahm.v14i5.2856', 'GC-MS (Durva Taila)'),
    ('Lauric acid', '10.70066/jahm.v14i5.2856', 'GC-MS (Durva Taila)'),
    ('Beta-sitosterol', '10.18231/j.jpbs.2023.018', 'GC-MS / HPTLC'),
    ('Quercetin', '10.1016/j.phyplu.2021.100020', 'HPLC'),
    ('Chlorogenic acid', '10.1016/j.phyplu.2021.100020', 'RP-HPLC'),
    ('Syringic acid', '10.18231/j.jpbs.2023.018', 'GC-MS'),
    ('Celecoxib', '10.1016/j.jep.2009.02.036', 'Positive Control (COX-2)'),
    ('Albendazole', '10.32463/rphs.2018.v04i03.13', 'Positive Control (Anthelmintic)')
]

rows = []
for name, doi, method in compounds:
    res = pcp.get_compounds(name, 'name')
    if res:
        c = res[0]
        smi = c.connectivity_smiles or c.SMILES
        mol = Chem.MolFromSmiles(smi)
        mw = round(Descriptors.MolWt(mol), 2)
        logp = round(Descriptors.MolLogP(mol), 2)
        tpsa = round(Descriptors.TPSA(mol), 2)
        hbd = Descriptors.NumHDonors(mol)
        hba = Descriptors.NumHAcceptors(mol)
        viol = sum([mw > 500, logp > 5, hbd > 5, hba > 10])
        rows.append({'compound_name': name, 'pubchem_cid': c.cid, 'smiles': smi, 'mol_weight': mw, 'logP': logp, 'TPSA': tpsa, 'H_donors': hbd, 'H_acceptors': hba, 'lipinski_violations': viol, 'how_found': method, 'source_doi': doi})

df = pd.DataFrame(rows)
df.to_csv('durva_compounds_v1_generated.csv', index=False)
df
